# Part 11 — CMIP6 delta-change future suitability & shift

Apply NEX-GDDP-CMIP6 monthly **change factors** (ratio for precip, additive Δ for tasmax/tasmin; ensemble of 5 GCMs) to the observed **TerraClimate 1991–2020** baseline, re-derive the climate bands with the *same* `features.derive_climate_bands`, and re-run the **Part-9 membership** on a future stack (terrain/soil/etc. static). Engine: `src/cmip6.py`.

Because baseline and future share one derivation, `suit_future − suit_present` is a **pure climate signal** (identity factors reproduce `feat_climate`). PET uses a Hargreaves ratio (Ra cancels); srad & soil-moisture held at baseline (documented caveats). Ensemble factors → `suit_future`; per-GCM signs → `agreement`.

**Output:** `suit_future_<ssp>_<window>`, `delta_<ssp>_<window>`, `agreement_<ssp>_<window>`. **DoD:** Δ + agreement render; hindcast bias bounded.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import ee, geemap
import utils, features
project = utils.init()
print('EE initialized; project =', project)
import cmip6, membership, external

In [ ]:
aoi = utils.load_aoi(project)
print('AOI area (km^2):', round(aoi.area(1000).divide(1e6).getInfo(), 1))

### Load present suitability + masks + side-image factors; pin the GCM ensemble
`extras` (the `sit_`/`rl_` side images) are **held at baseline** in the future exactly like srad/soil-moisture — clearness keys on baseline srad, seasonal-water distance is static, native cover is a present observation — so Δ isolates the climate-driven factors (incl. solar's new `clim_twarm_q` heat de-rating, which *does* move with the future stack). The same `extras` feed present-rederived and future so they cancel in Δ.

In [ ]:
suit_present = ee.Image(utils.asset_id(project, 'suit_present'))
lc = ee.Image(utils.asset_id(project, 'feat_landcover'))
seg_cfg = utils.cfg('segments'); segs = list(seg_cfg['segments'])
siting = ee.Image(utils.asset_id(project, 'feat_siting'))
consv = ee.Image(utils.asset_id(project, 'feat_conservation'))
extras = {'sit_': siting, 'rl_': external.realized_features(aoi), 'cv_': consv}
cc = utils.cfg()['cmip6']
models, scenarios, windows = cc['models'], cc['scenarios'], cc['windows']
print(f'{len(models)} GCMs:', models)
print('scenarios:', scenarios, '| windows:', list(windows))
# observed TerraClimate monthly climatology — built once, reused for every (ssp, window)
base = cmip6.baseline_monthly(aoi)
# Δ reference: present recomputed on-the-fly so the baseline cancels exactly in Δ.
# (Subtracting the *exported* suit_present asset injects an export-reproject offset;
#  verified: identity factors give Δ≈0 vs this, but spurious ±0.05–0.3 vs the asset.)
suit_present_re = cmip6.suit_present_rederived(project, aoi, lc, seg_cfg, extras=extras)

### Hindcast sanity — CMIP6 historical vs observed baseline bias
Delta-change *cancels* this bias, so it is a report (should be bounded/plausible, not zero), not part of the projection. Heavier reduction — skip under tight quota.

In [ ]:
import pandas as pd
hb = cmip6.hindcast_bias(models, aoi).getInfo()
print('CMIP6 historical − observed baseline (annual precip mm, mean temp °C):')
print(pd.Series(hb).round(2))

### Future suitability + ΔSuitability — one scenario/window (detail)
`suit_future` is one membership run on the ensemble-mean future climate.

In [ ]:
ssp, win_name = 'ssp245', '2031_2050'  # windows[win_name] is the [start,end] list cmip6 needs
sf = cmip6.suit_future(project, models, ssp, windows[win_name], aoi, lc, seg_cfg, base=base, extras=extras)
dl = cmip6.delta(sf, suit_present_re, segs)
print('suit_future bands:', sf.bandNames().getInfo())

### DoD sanity — ΔSuitability range per segment (expect small, signed values)

In [ ]:
utils.range_report(dl, aoi)

### Quick look — ΔSuitability (soybean), diverging palette

In [ ]:
Map = geemap.Map(); Map.centerObject(aoi, 7)
dvis = {'min': -0.2, 'max': 0.2, 'palette': ['#b2182b','#f7f7f7','#2166ac']}
Map.addLayer(dl.select('delta_soybean'), dvis, f'Δ soybean {ssp} {win}')
Map.addLayer(dl.select('delta_cattle'), dvis, f'Δ cattle {ssp} {win}', False)
Map.addLayer(aoi, {}, 'AOI', False)
Map

### Export `suit_future_*` + `delta_*` for every scenario × window
Batch tasks (heavy: each rebuilds the CMIP6 monthly climatologies server-side). Monitor with `earthengine task list`.

In [ ]:
utils.ensure_folder(project)
tasks = []
for ssp in scenarios:
    for win_name, win in windows.items():  # win is [start,end]; win_name is the asset suffix
        sf = cmip6.suit_future(project, models, ssp, win, aoi, lc, seg_cfg, base=base, extras=extras)
        dl = cmip6.delta(sf, suit_present_re, segs)
        tasks.append(utils.export_image(sf, project, f'suit_future_{ssp}_{win_name}', aoi))
        tasks.append(utils.export_image(dl, project, f'delta_{ssp}_{win_name}', aoi))
print(utils.task_summary(tasks))

### (Optional) Ensemble-agreement maps — fraction of GCMs agreeing on sign of Δ
Confidence layer; **heavier** (one membership run per GCM). Submit per scenario/window as quota allows; coarsen if a task times out.

In [ ]:
ag_tasks = []
for ssp in scenarios:
    for win_name, win in windows.items():
        ag = cmip6.agreement(project, models, ssp, win, aoi, suit_present_re, lc, seg_cfg, base=base, extras=extras)
        ag_tasks.append(utils.export_image(ag, project, f'agreement_{ssp}_{win_name}', aoi))
print(utils.task_summary(ag_tasks))